In [ ]:
pip install langchain langchain-google-genai langchain-community pypdf faiss-cpu google-generativeai


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.5/503.5 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.8 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.18
    Uninstalling langchain-core-1.2.18:
      Successfully uninstalled langchain-core-1.2.18
ERROR: pip's dependen

In [ ]:
import langchain
import faiss
from langchain_google_genai import ChatGoogleGenerativeAI
print("All imports successful ✅")

All imports successful ✅


In [ ]:
from google.colab import userdata
import os

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
print(os.environ["GOOGLE_API_KEY"][:10])  # prints first 10 chars to verify

AIzaSyApCU


In [ ]:
from google.colab import files

uploaded = files.upload()  # A dialog box will pop up — choose your PDF
file_name = list(uploaded.keys())[0]
print(f"Uploaded: {file_name} ✅")

Saving finalfinal_srs.pdf to finalfinal_srs.pdf
Uploaded: finalfinal_srs.pdf ✅


In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os

# Fix filename with spaces
safe_name = "uniqfinalsrs.pdf"
os.rename(file_name, safe_name)

# Load PDF
loader = PyPDFLoader(safe_name)
pages = loader.load()
print(f"Total pages loaded: {len(pages)}")

# Split into chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(pages)
print(f"Total chunks created: {len(chunks)} ✅")

Total pages loaded: 17
Total chunks created: 80 ✅


In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS

# Create embeddings using Gemini
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

# Store in FAISS vector database
vector_store = FAISS.from_documents(chunks, embeddings)
print("Vector store created ✅")

Vector store created ✅


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.3)

prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant. Answer the question based only on the context below.

Previous conversation:
{chat_history}

Context from PDF:
{context}

Question: {question}
""")

chain = prompt | llm | StrOutputParser()
chat_history = []

print("Chatbot ready ✅")

Chatbot ready ✅


In [ ]:
print("📄 PDF Chatbot is ready! Type 'exit' to stop.\n")

while True:
    question = input("You: ")
    if question.lower() == "exit":
        print("Goodbye! 👋")
        break

    # Find relevant chunks
    docs = vector_store.similarity_search(question, k=3)

    # Get page numbers
    pages_used = set([doc.metadata.get("page", "?") + 1 for doc in docs])
    context = "\n\n".join([doc.page_content for doc in docs])

    # Format chat history
    history_text = "\n".join([f"Human: {q}\nAssistant: {a}" for q, a in chat_history])

    # Get answer
    answer = chain.invoke({
        "context": context,
        "question": question,
        "chat_history": history_text
    })

    # Save to history
    chat_history.append((question, answer))

    # Print answer + sources
    print(f"\nBot: {answer}")
    print(f"📖 Source pages: {sorted(pages_used)}\n")

📄 PDF Chatbot is ready! Type 'exit' to stop.

You: what is srs?

Bot: SRS stands for Software Requirements Specification.
📖 Source pages: [1, 4, 16]

You: can you say more about it?

Bot: I apologize, but the provided context does not offer additional information about what a Software Requirements Specification (SRS) is. The context describes features and functionalities of a system called ItemFlow, such as its dashboard, reporting, AI assistant, and low stock alerts.
📖 Source pages: [2, 5, 12]

You: why is it useful?

Bot: It is recommended to read the complete SRS document carefully because it contains detailed information about the system’s functional and nonfunctional requirements.
📖 Source pages: [4, 5]

You: exit
Goodbye! 👋


In [ ]:
!pip install langchain-classic -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.5/503.5 kB 24.9 MB/s eta 0:00:00


import google.generativeai as genai
import os

genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

# List all available models
for m in genai.list_models():
    if "embedContent" in m.supported_generation_methods:
        print(m.name)

In [ ]:
import google.generativeai as genai
import os

genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

for m in genai.list_models():
    if "generateContent" in m.supported_generation_methods:
        print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-2.5-flash-lite-preview-09-2025
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-robotics-er-1.5-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-research-pro-preview-12-2025
